# Bài Toán 6 Con Hậu — Thuật Toán Min-Conflict

**Bài toán:** Đặt **6 con hậu** lên bàn cờ **6×6** sao cho **không con nào ăn được con nào**
(không cùng hàng, không cùng cột, không cùng đường chéo).

**Thuật toán:** **Min-Conflict** (Cực tiểu xung đột) — một thuật toán tìm kiếm cục bộ (local search)
dựa trên heuristic sửa chữa lặp (iterative repair).

**Ý tưởng chính:**
1. Đặt ngẫu nhiên mỗi cột đúng 1 con hậu (đảm bảo không xung đột cột).
2. Chọn ngẫu nhiên một con hậu đang bị xung đột.
3. Di chuyển nó đến hàng trong cùng cột có **ít xung đột nhất**.
4. Lặp lại cho đến khi không còn xung đột nào (hoặc quá số bước tối đa).

**Đặc điểm:** Rất nhanh với bài toán N-Queens (thường giải được N=1,000,000+ trong ~50 bước).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import time

plt.rcParams['font.family'] = 'Tahoma'

## 1. Hàm tiện ích — Tính xung đột & Vẽ bàn cờ

In [ ]:
N = 6  # Kích thước bàn cờ (6 con hậu trên bàn cờ 6×6)


def count_conflicts(state, col, row):
    """Đếm số xung đột nếu đặt hậu ở cột `col` tại hàng `row`."""
    conflicts = 0
    for c in range(N):
        if c == col:
            continue
        r = state[c]
        # Cùng hàng hoặc cùng đường chéo
        if r == row or abs(r - row) == abs(c - col):
            conflicts += 1
    return conflicts


def total_conflicts(state):
    """Tổng số cặp hậu xung đột trên toàn bàn cờ."""
    total = 0
    for c in range(N):
        total += count_conflicts(state, c, state[c])
    return total // 2  # Mỗi cặp đếm 2 lần


def find_conflicted_queens(state):
    """Trả về danh sách các cột có hậu đang bị xung đột."""
    conflicted = []
    for c in range(N):
        if count_conflicts(state, c, state[c]) > 0:
            conflicted.append(c)
    return conflicted


def draw_board(state, highlight_col=None, title='', show_conflicts=True):
    """Vẽ bàn cờ với các con hậu."""
    fig, ax = plt.subplots(figsize=(5.5, 5.5))

    # Vẽ ô bàn cờ
    for r in range(N):
        for c in range(N):
            color = '#F0D9B5' if (r + c) % 2 == 0 else '#B58863'
            rect = plt.Rectangle((c, N - 1 - r), 1, 1, facecolor=color,
                                edgecolor='none', zorder=1)
            ax.add_patch(rect)

    # Vẽ hậu
    for c in range(N):
        r = state[c]
        y = N - 1 - r

        # Highlight cột đang xét
        if c == highlight_col:
            highlight = plt.Rectangle((c, 0), 1, N, facecolor='#FFEB3B',
                                     edgecolor='none', alpha=0.5, zorder=2)
            ax.add_patch(highlight)

        # Vẽ ô xung đột (nếu có)
        if show_conflicts and c != highlight_col:
            hr = state[highlight_col] if highlight_col is not None else -1
            if r == hr or (highlight_col is not None and abs(r - hr) == abs(c - highlight_col)):
                conflict_overlay = plt.Rectangle((c, y), 1, 1, facecolor='#FF6B6B',
                                                edgecolor='none', alpha=0.45, zorder=3)
                ax.add_patch(conflict_overlay)

        # Hậu
        ax.text(c + 0.5, y + 0.5, '♛', ha='center', va='center',
               fontsize=26, color='black', zorder=5)

    # Highlight viền cột đang chọn
    if highlight_col is not None:
        border = plt.Rectangle((highlight_col, 0), 1, N, facecolor='none',
                              edgecolor='#FF9800', linewidth=3, zorder=6)
        ax.add_patch(border)

    # Cấu hình trục
    ax.set_xlim(0, N)
    ax.set_ylim(0, N)
    ax.set_xticks([i + 0.5 for i in range(N)])
    ax.set_yticks([i + 0.5 for i in range(N)])
    ax.set_xticklabels([str(i) for i in range(N)])
    ax.set_yticklabels([str(N - 1 - i) for i in range(N)])
    ax.set_xlabel('Cột', fontsize=10)
    ax.set_ylabel('Hàng', fontsize=10)
    ax.set_aspect('equal')

    conflicts = total_conflicts(state)
    full_title = f'{title}'
    if show_conflicts:
        full_title += f'  |  Xung đột: {conflicts}'
    ax.set_title(full_title, fontsize=12, fontweight='bold', pad=8)

    plt.tight_layout()
    plt.show()

In [ ]:
def min_conflict(state, max_steps=1000, visualize=True):
    """Thuật toán Min-Conflict cho bài toán N-Queens."""
    step = 0
    viz_every = max(1, max_steps // 12)  # Hiển thị ~12 bước

    if visualize:
        print(f"Trạng thái ban đầu (bố trí ngẫu nhiên):")
        print(f"  State = {list(state)}  (state[c] = hàng của hậu tại cột c)")
        print(f"  Tổng xung đột: {total_conflicts(state)}")
        draw_board(state, title=f'Bước 0: Bố trí ban đầu')

    for step in range(1, max_steps + 1):
        # 1. Tìm các hậu đang bị xung đột
        conflicted = find_conflicted_queens(state)

        # 2. Nếu không còn xung đột → đã tìm được lời giải
        if not conflicted:
            if visualize:
                print(f"\n✅ GIẢI ĐƯỢC sau {step - 1} bước!")
            return True, step - 1

        # 3. Chọn ngẫu nhiên một hậu bị xung đột
        col = random.choice(conflicted)
        old_row = state[col]

        # 4. Đánh giá từng hàng trong cột này → chọn hàng ít xung đột nhất
        row_scores = []
        for row in range(N):
            conflicts = count_conflicts(state, col, row)
            row_scores.append((conflicts, row))

        # Tìm xung đột nhỏ nhất
        min_conf = min(s[0] for s in row_scores)
        best_rows = [s[1] for s in row_scores if s[0] == min_conf]

        # 5. Chọn ngẫu nhiên trong số các hàng tốt nhất
        new_row = random.choice(best_rows)

        # 6. Di chuyển hậu
        state[col] = new_row

        # Hiển thị
        if visualize and (step <= 6 or step % viz_every == 0):
            verb = 'Di chuyển'
            if old_row == new_row:
                verb = 'Giữ nguyên'
            marker = ' ← MIN' if min_conf == 0 else ''
            print(f"\nBước {step}: {verb} hậu cột {col}: hàng {old_row} → {new_row}")
            print(f"  Các hàng ít xung đột nhất: {best_rows} (xung đột={min_conf}){marker}")
            print(f"  Còn {len(conflicted)} hậu bị xung đột")
            draw_board(state, highlight_col=col,
                      title=f'Bước {step}: Hậu cột {col} → hàng {new_row}')

    # Hết số bước mà chưa giải được → khởi động lại
    if visualize:
        print(f"\n⚠ Chưa giải được sau {max_steps} bước — cần khởi động lại với bố trí khác.")
    return False, max_steps

## 2. Chạy thuật toán — Minh họa từng bước

In [ ]:
# ── Khởi tạo & Chạy Min-Conflict ──

random.seed(42)  # Cố định seed để kết quả có thể tái lập

# Mỗi cột có đúng 1 hậu, đặt ngẫu nhiên vào 1 hàng
initial_state = [random.randint(0, N - 1) for _ in range(N)]

print("=" * 55)
print(f"BÀI TOÁN {N} CON HẬU — THUẬT TOÁN MIN-CONFLICT")
print(f"Bàn cờ: {N}×{N}  |  Số hậu: {N}")
print("=" * 55)

state = list(initial_state)
success, steps = min_conflict(state, max_steps=200)

if success:
    print("\n" + "=" * 55)
    print("✨ KẾT QUẢ CUỐI CÙNG")
    print("=" * 55)
    print(f"  Bố trí hậu: {list(state)}")
    print(f"  Tổng xung đột: {total_conflicts(state)}")
    print(f"  Số bước thực hiện: {steps}")
    draw_board(state, title='Lời giải cuối cùng: 6 hậu không ăn nhau')

## 3. Phân tích — Đánh giá hiệu năng

In [ ]:
# ── Thống kê qua nhiều lần chạy ──

def solve_n_queens(n, trials=50, max_steps=1000):
    """Giải bài toán N-Queens nhiều lần để thống kê."""
    success_count = 0
    steps_list = []
    restarts = 0

    for t in range(trials):
        state = [random.randint(0, n - 1) for _ in range(n)]
        success, steps = min_conflict(state, max_steps=max_steps, visualize=False)

        if success:
            success_count += 1
            steps_list.append(steps)
        else:
            restarts += 1
            # Thử lại với bố trí mới (thường chỉ cần 1-2 lần restart)
            state = [random.randint(0, n - 1) for _ in range(n)]
            success2, steps2 = min_conflict(state, max_steps=max_steps, visualize=False)
            if success2:
                success_count += 1
                steps_list.append(steps2)
            else:
                restarts += 1

    return success_count, steps_list, restarts


print("Đánh giá Min-Conflict trên bài toán 6-Queens...")
random.seed(123)
success, steps_list, restarts = solve_n_queens(N, trials=30)

print(f"\nKết quả qua 30 lần chạy:")
print(f"  Tỉ lệ thành công: {success}/30 ({100*success/30:.0f}%)")
if steps_list:
    print(f"  Số bước trung bình: {sum(steps_list)/len(steps_list):.1f}")
    print(f"  Số bước ít nhất:    {min(steps_list)}")
    print(f"  Số bước nhiều nhất:  {max(steps_list)}")
print(f"  Số lần restart:     {restarts}")

# So sánh với các kích thước khác
print(f"\n{'─' * 45}")
print(f"So sánh các kích thước bàn cờ khác nhau:")
print(f"{'─' * 45}")
for size in [4, 6, 8, 10, 20, 50]:
    s, steps_l, _ = solve_n_queens(size, trials=10, max_steps=2000)
    avg_steps = sum(steps_l) / len(steps_l) if steps_l else 0
    print(f"  N={size:3d}: thành công {s}/10, trung bình {avg_steps:6.1f} bước")

## 4. Tổng kết

| Khía cạnh | Mô tả |
|-----------|-------|
| **Bài toán** | 6-Queens: đặt 6 hậu trên bàn 6×6 không ăn nhau |
| **Thuật toán** | Min-Conflict — tìm kiếm cục bộ, sửa chữa lặp |
| **Heuristic** | Chọn hàng có ít xung đột nhất trong cột |
| **Khởi tạo** | Ngẫu nhiên, mỗi cột 1 hậu |
| **Độ phức tạp** | O(max_steps × N²) mỗi bước |

**Ưu điểm:**
- Cực kỳ nhanh với N-Queens (hàng triệu hậu chỉ trong ~50 bước)
- Đơn giản, dễ cài đặt
- Hiệu quả với nhiều bài toán CSP thực tế (lập lịch, phân công)

**Nhược điểm:**
- Không đảm bảo tìm được lời giải (có thể kẹt ở cực tiểu cục bộ)
- Cần restart nếu thất bại

**Cơ chế thoát cực tiểu cục bộ:** Chọn ngẫu nhiên trong các hàng tốt nhất giúp thuật toán
có cơ hội thoát khỏi plateau (vùng bằng phẳng) và tiếp tục tìm kiếm.